In [ ]:
!pip install transformers

In [ ]:
from transformers import ViTForImageClassification, Trainer, TrainingArguments, AutoImageProcessor
from PIL import Image
import torch
import pandas as pd
import os
from torch.utils.data import Dataset, random_split

In [ ]:
model_name = "google/vit-base-patch16-224"
model = ViTForImageClassification.from_pretrained(model_name, num_labels=2,ignore_mismatched_sizes=True)
feature_extractor = AutoImageProcessor.from_pretrained(model_name)

class HouseDataset(Dataset):
    def __init__(self, df, image_folder, feature_extractor):
        self.df = df
        self.image_folder = image_folder
        self.feature_extractor = feature_extractor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_path = os.path.join(self.image_folder, self.df.iloc[idx]["image_name"])
        image = Image.open(image_path).convert("RGB")
        inputs = self.feature_extractor(images=image, return_tensors="pt")
        label = torch.tensor(self.df.iloc[idx]["class"], dtype=torch.long)
        return {"pixel_values": inputs["pixel_values"].squeeze(), "labels": label}

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224
Key               | Status   |                                                                                        
------------------+----------+----------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000, 768]) vs model:torch.Size([2, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch ckpt: torch.Size([1000]) vs model:torch.Size([2])          

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/super-ai-engineer-season-6-individual-hackathon-house-recognition/train.csv")
image_folder = "/content/drive/MyDrive/super-ai-engineer-season-6-individual-hackathon-house-recognition/train/train" # Corrected path
dataset = HouseDataset(df, image_folder, feature_extractor)

In [ ]:
dataset

In [ ]:
train_size = int(0.8 * len(dataset))
eval_size = len(dataset) - train_size
train_dataset, eval_dataset = random_split(dataset, [train_size, eval_size])

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=4,
    weight_decay=0.01,
    report_to="none",
)

In [ ]:
from sklearn.metrics import accuracy_score
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=1)
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc}

model.to(device) # Explicitly move model to device

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.260130,0.192366,0.939086
2,0.110629,0.140102,0.959391
3,0.032424,0.141116,0.954315
4,0.016148,0.147322,0.956007


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2364, training_loss=0.09058618565703003, metrics={'train_runtime': 1613.1206, 'train_samples_per_second': 5.857, 'train_steps_per_second': 1.465, 'total_flos': 7.321443178786652e+17, 'train_loss': 0.09058618565703003, 'epoch': 4.0})

In [ ]:
output_model_dir = "./my_trained_model"
model.save_pretrained(output_model_dir)
feature_extractor.save_pretrained(output_model_dir)
print(f"Model and feature extractor saved to {output_model_dir}")

In [ ]:
import os
directory_path = '/content/drive/MyDrive/super-ai-engineer-season-6-individual-hackathon-house-recognition/train/'

if os.path.exists(directory_path):
    contents = os.listdir(directory_path)
    print(f"Contents of '{directory_path}':")
    # Display first 10 items if there are many
    if len(contents) > 10:
        for item in contents[:10]:
            print(item)
        print(f"... and {len(contents) - 10} more items.")
    else:
        for item in contents:
            print(item)
else:
    print(f"Directory not found: {directory_path}")

Contents of '/content/drive/MyDrive/super-ai-engineer-season-6-individual-hackathon-house-recognition/train/':
train


In [ ]:
def predict(images_folder, model, feature_extractor, device):
    model.eval()
    predictions = []
    image_files = [f for f in os.listdir(images_folder) if f.endswith(".jpg") or f.endswith(".png")]

    for image_file in image_files:
        image_path = os.path.join(images_folder, image_file)
        image = Image.open(image_path).convert("RGB")
        inputs = feature_extractor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            predicted_label = torch.argmax(outputs.logits, dim=-1).item()
        predictions.append((image_file, predicted_label))

    return predictions

In [ ]:
test_folder = "/content/drive/MyDrive/super-ai-engineer-season-6-individual-hackathon-house-recognition/test/test"
predictions = predict(test_folder, model, feature_extractor,device)

In [ ]:
predictions

[('5b22e513.jpg', 1),
 ('7e78ba7d.jpg', 1),
 ('7045ee0e.jpg', 1),
 ('583e01c9.jpg', 0),
 ('5bcca9b9.jpg', 1),
 ('5c421eed.jpg', 1),
 ('69e54e88.jpg', 0),
 ('8176d497.jpg', 1),
 ('6cf50ba4.jpg', 1),
 ('5aca8314.jpg', 1),
 ('72ce7f51.jpg', 1),
 ('5c43fc0d.jpg', 0),
 ('6d626e6c.jpg', 0),
 ('6976bdce.jpg', 0),
 ('77770040.jpg', 0),
 ('60aefb64.jpg', 0),
 ('67d766d2.jpg', 1),
 ('655d8bff.jpg', 0),
 ('673b02ea.jpg', 0),
 ('7a101d27.jpg', 0),
 ('832900d5.jpg', 0),
 ('711bde86.jpg', 0),
 ('645e314f.jpg', 0),
 ('7b5e37f5.jpg', 1),
 ('7e8862e8.jpg', 0),
 ('71dd0a48.jpg', 1),
 ('78fb392a.jpg', 1),
 ('81bc175f.jpg', 0),
 ('806357e2.jpg', 1),
 ('7d7a3d8c.jpg', 0),
 ('754a112d.jpg', 1),
 ('791982df.jpg', 0),
 ('74d21a05.jpg', 1),
 ('7b304617.jpg', 0),
 ('5f7a930c.jpg', 1),
 ('62784888.jpg', 1),
 ('582c787c.jpg', 0),
 ('5f033aaf.jpg', 1),
 ('7c112068.jpg', 0),
 ('5e8aea48.jpg', 0),
 ('6e0cabd5.jpg', 1),
 ('7ee9ab15.jpg', 0),
 ('83f20d72.jpg', 0),
 ('71032a54.jpg', 0),
 ('75babc9b.jpg', 1),
 ('73729ab

In [ ]:
submit = pd.read_csv("/content/drive/MyDrive/super-ai-engineer-season-6-individual-hackathon-house-recognition/sample_submission.csv")

In [ ]:
submit

,id,answer
0,e4b420b0,0.0
1,23efa479,0.0
2,1f0f2402,0.0
3,8a60480c,NaN
4,11f20127,NaN
...,...,...
1545,ed44ac4d,NaN
1546,a2067109,NaN
1547,1481178a,NaN
1548,340a3e9d,NaN


In [ ]:
ss = pd.DataFrame(predictions)

In [ ]:
ss

,0,1
0,5b22e513.jpg,1
1,7e78ba7d.jpg,1
2,7045ee0e.jpg,1
3,583e01c9.jpg,0
4,5bcca9b9.jpg,1
...,...,...
1545,58d343f2.jpg,1
1546,6a5a68ee.jpg,0
1547,7ba6d581.jpg,0
1548,5d871084.jpg,0


In [ ]:
ss = ss.rename(columns={0: "id", 1: "answer"})
ss["id"] = ss["id"].str.replace(".jpg", "")

In [ ]:
submit = submit.set_index("id")
ss = ss.set_index("id")
df_sorted = ss.reindex(submit.index)

In [ ]:
df_sorted

,answer
id,
e4b420b0,0
23efa479,0
1f0f2402,0
8a60480c,0
11f20127,0
...,...
ed44ac4d,1
a2067109,1
1481178a,1


In [ ]:
df_sorted.to_csv("submit_house.csv")